# Preps

In [1]:
# Grounding
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/zeguru/Projects/AI/tiny-text-transformers


In [17]:
#Imports
from pathlib import Path

import torch
import json
import random

from datetime import datetime
from src.infra.dataset.util import split_tokens, get_batch
from src.infra.tokenizer.character import CharacterTokenizer
from src.infra.tokenizer.bpe import BPETokenizer
from src.infra.dataset.answering import QuestionAnsweringDataset
from torch.utils.data import random_split

from src.task.config import ModelConfig,TrainingConfig

from src.task.model import TinyAnsweringModel
from src.task.training.question_answering import train
from src.task.training.util import TrainingLogger, count_parameters, separator


# Configs

In [37]:
#Model + Training Config

torch.manual_seed(42)

logger = TrainingLogger("logs/question_answering.log")


# Model hyper params
model_config = ModelConfig(
    d_model=128, 
    number_of_heads=2, 
    context_length=1024
    )

# Training settings
training_config = TrainingConfig(
    num_steps=100,
    eval_interval=10,
    batch_size=64
    )

In [5]:
#Paths
MODEL_DESTINATION = "tiny_answering.pt"

SQUAD_PATH = Path("../data/squad/dev-v1.1.json")

CHAR_TOKENIZER_PATH = Path("../checkpoints/qa_tokenizer.json")
BPE_TOKENIZER_PATH = Path("../checkpoints/bpe_tokenizer.json")

print(f"Squad file: {SQUAD_PATH}")
print(f"Tokenizer file:  {CHAR_TOKENIZER_PATH}")
print(f"Tokenizer file:  {BPE_TOKENIZER_PATH}")

Squad file: ../data/squad/dev-v1.1.json
Tokenizer file:  ../checkpoints/qa_tokenizer.json
Tokenizer file:  ../checkpoints/bpe_tokenizer.json


# Corpus

In [7]:
# Load
with SQUAD_PATH.open("r", encoding="utf-8") as file:
    squad = json.load(file)

# payload
records = []

In [30]:
# Load subset of the corpus

LOAD_ALL_DATA = False  # Change to True when you need everything


records = []

if LOAD_ALL_DATA:
    articles = squad["data"]  # All articles
else:
    articles = [
        article
        for article in squad["data"]
        if article["title"] == "Kenya"
    ]  # Sample: Kenya only

# Load them
for article in articles:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            for answer in qa["answers"]:
                records.append({
                    "question": qa["question"],
                    "context": context,
                    "answer": answer["text"],
                    "answer_start": answer["answer_start"],
                })
                
print(f"Number of records: {len(records)}")

num = random.randint(1, len(records))

print(f"\n=== Sample record: {num} ===")

sample = records[num]
question = sample["question"]
context = sample["context"] 
answer_text = sample["answer"]
answer_start = sample["answer_start"]
answer_end = answer_start + len(answer_text)

print("Context:", context)
print("Question:", question)
print("Answer:", answer_text)
print("Answer Start:", answer_start)
print("Answer End:", answer_end)

Number of records: 691

=== Sample record: 646 ===
Context: Kenya won several medals during the Beijing Olympics, six gold, four silver and four bronze, making it Africa's most successful nation in the 2008 Olympics. New athletes gained attention, such as Pamela Jelimo, the women's 800m gold medalist who went ahead to win the IAAF Golden League jackpot, and Samuel Wanjiru who won the men's marathon. Retired Olympic and Commonwealth Games champion Kipchoge Keino helped usher in Kenya's ongoing distance dynasty in the 1970s and was followed by Commonwealth Champion Henry Rono's spectacular string of world record performances. Lately, there has been controversy in Kenyan athletics circles, with the defection of a number of Kenyan athletes to represent other countries, chiefly Bahrain and Qatar. The Kenyan Ministry of Sports has tried to stop the defections, but they have continued anyway, with Bernard Lagat the latest, choosing to represent the United States. Most of these defections occu

In [35]:
# Tokenizer Init

char_tokenizer = CharacterTokenizer.load(CHAR_TOKENIZER_PATH)
bpe_tokenizer = BPETokenizer.load(BPE_TOKENIZER_PATH)

#Choose the tokenizer... manual for now
tokenizer = char_tokenizer

vocab_size = tokenizer.vocab_size

print(f"vocabulary size: {vocab_size}: \ntokenizer: {tokenizer}")

vocabulary size: 98: 
tokenizer: <src.infra.tokenizer.character.CharacterTokenizer object at 0x11baec2f0>


## Dataset preps

In [38]:
#
dataset = QuestionAnsweringDataset(
    records=records,
    tokenizer=tokenizer,
    max_length=model_config.context_length,         #to manage truncation and padding
    )


n = len(dataset)
train_size = int(0.90 * n)
val_size = int(0.05 * n)
test_size = n - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
    )

logger.log(f"Train examples: {len(train_dataset):,}")
logger.log(f"Val examples:   {len(val_dataset):,}")
logger.log(f"Test examples:  {len(test_dataset):,}")

Train examples: 613
Val examples:   34
Test examples:  35
